# Softcite-style analysis of HAL software mentions

This notebook reproduces the spirit of the [`softcite/softcite-extractions-oa`](https://github.com/softcite/softcite-extractions-oa) analyses against software mentions extracted from HAL papers: the raw [Softcite `software-mentions`](https://github.com/softcite/software-mentions) output for the PEPR Santé Numérique corpus.

Softcite ships its dataset as **three tables** — `papers`, `mentions`, `purpose_assessments` — and demonstrates analyses on top of them. We rebuild those same three tables here as in-memory **pandas DataFrames** by flattening the raw JSON, then run an exploratory analysis.

**Scope choices (locked in):**
- We use the Softcite output files (`*.software.json`). Most were run on the GROBID TEI and also carry bibliographic metadata (`biblio`).
- Tables stay **in memory** — no Parquet export.
- Sections 2–6 are about the **explicit** software: software named in the text (Softcite `software-type` `software`, `component` or `environment`), counted **once per paper by its normalized name** (`normalizedForm`), and characterized at **document level** (Softcite's `documentContextAttributes`: used / created / shared, judged over the whole paper). Section 7 looks at the **implicit** mentions (`software-type` `implicit`: "our code", "the scripts").

**Known limitation:** the raw Softcite output has no `published_year`, `genre`, or `license` — only `biblio` (title / authors / doi). So Softcite's *papers-per-year trend* example cannot be reproduced; we pivot to the dimensions that are present (software names, purpose × scope, language, publisher, version, URL, citations). See the final **Limitations** section.

> **Hackathon note.** This notebook comes from [Inria-Datalake/analysis-notebooks](https://github.com/Inria-Datalake/analysis-notebooks), adapted to the PEPR Santé Numérique corpus.
>
> **Input:** the Softcite `software-mentions` output for the PEPR corpus, which is already on 🤗 [`HackatonSanteNumerique/pepr-hal-data`](https://huggingface.co/datasets/HackatonSanteNumerique/pepr-hal-data) in `softcite/`. Download it first (about 12 MB):
>
> ```bash
> uv run python scripts/download_pepr.py --pattern "softcite/*"
> ```
>
> The notebook then reads `data/pepr-hal-data/softcite/*.software.json`: 265 papers, 211 of them with at least one explicit software mention and 122 with an implicit one.
>
> The same three tables (`papers`, `mentions`, `purpose_assessments`) can be built from other extractors (CoPaLink, Baguette…) if you map their output to the schema below. That's a good way to compare models.

## 0 — Setup

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Softcite output lives in <repo>/data/pepr-hal-data/softcite; this notebook lives in notebooks/.
# Resolve it whether the kernel runs from notebooks/ or the repo root.
DATA_DIR = Path("data/pepr-hal-data/softcite")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data/pepr-hal-data/softcite")
assert DATA_DIR.exists(), (
    "Could not find data/pepr-hal-data/softcite/ — run: uv run python scripts/download_pepr.py --pattern 'softcite/*'"
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# --- chart style -------------------------------------------------------------
# Plotly charts: hover a bar for its exact value. "light" or "dark": each has its own steps of the
# same palette (categorical slots in a fixed, colorblind-checked order) and chrome.
THEME = "light"
STYLE = {
    "light": dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
                  series=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]),
    "dark": dict(surface="#1a1a19", ink="#f0efec", ink2="#c3c2b7", muted="#898781", grid="#2c2c2a", axis="#383835",
                 series=["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181", "#008300", "#9085e9", "#e66767"]),
}[THEME]
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
_axis = dict(showgrid=True, gridcolor=STYLE["grid"], gridwidth=1, showline=True, linecolor=STYLE["axis"], linewidth=1,
             zeroline=False, ticks="", tickfont=dict(color=STYLE["muted"]),
             title=dict(font=dict(color=STYLE["ink2"])), automargin=True)
pio.templates["pepr"] = go.layout.Template(layout=dict(
    font=dict(family=FONT, size=13, color=STYLE["ink"]),
    paper_bgcolor=STYLE["surface"], plot_bgcolor=STYLE["surface"], colorway=STYLE["series"],
    title=dict(font=dict(size=16, color=STYLE["ink"]), x=0, xanchor="left", xref="paper"),
    xaxis=_axis, yaxis=_axis,
    legend=dict(font=dict(color=STYLE["ink2"]), title=dict(text="")),
    hoverlabel=dict(bgcolor=STYLE["surface"], bordercolor=STYLE["axis"], font=dict(family=FONT, color=STYLE["ink"])),
    barcornerradius=4, bargap=0.35, margin=dict(l=16, r=48, t=64, b=48),
))
pio.templates.default = "pepr"
# Interactive in JupyterLab and VS Code (Plotly mimetype) and in the classic notebook (HTML).
pio.renderers.default = "plotly_mimetype+notebook_connected"


def _bars(values, unit, fmt, suffix, labels):
    """One series of horizontal bars, the largest at the top; the exact value is in the tooltip."""
    values = values[::-1]
    return go.Bar(
        x=values.values, y=[str(k) for k in values.index], orientation="h", marker_color=STYLE["series"][0],
        text=[f"{v:{fmt}}{suffix}" for v in values.values] if labels else None,
        textposition="outside", textfont=dict(color=STYLE["ink2"]), cliponaxis=False, showlegend=False,
        hovertemplate=f"<b>%{{y}}</b><br>%{{x:{fmt}}}{suffix} {unit}<extra></extra>")


def hbar(values, title, unit, fmt=",d", suffix="", labels=False, x_title=None):
    fig = go.Figure(_bars(values, unit, fmt, suffix, labels))
    fig.update_layout(title=title, xaxis_title=x_title or unit, height=max(210, 28 * len(values) + 120))
    fig.update_yaxes(showgrid=False, tickfont=dict(color=STYLE["ink2"]))
    return fig


def hbar_pair(left, right, titles, units, fmt=",d"):
    """Two rankings side by side, e.g. by mentions and by papers."""
    fig = make_subplots(rows=1, cols=2, subplot_titles=titles, horizontal_spacing=0.3)
    for col, (values, unit) in enumerate(((left, units[0]), (right, units[1])), start=1):
        fig.add_trace(_bars(values, unit, fmt, "", False), row=1, col=col)
        fig.update_xaxes(title_text=unit, row=1, col=col)
    fig.update_yaxes(showgrid=False, tickfont=dict(color=STYLE["ink2"]))
    fig.update_annotations(font=dict(size=14, color=STYLE["ink"]))
    fig.update_layout(height=max(260, 28 * max(len(left), len(right)) + 140))
    return fig


print("Data directory:", DATA_DIR.resolve())

Data directory: /data/workspace/lfoppiano/inria/hackaton-sante-numerique-2026/data/pepr-hal-data/softcite


## 1 — Load &amp; flatten into the three Softcite tables

Each `*.software.json` file looks like:

```jsonc
{
  "id": "<hash>",
  "biblio": { "title": ..., "authors": ..., "doi": ... },   // optional
  "mentions": [
    {
      "software-name": { "rawForm": ..., "normalizedForm": ... },
      "version":   { "rawForm": ..., "normalizedForm": ... },  // optional
      "publisher": { ... },                                    // optional
      "language":  { ..., "wikidataId": ... },                 // optional
      "url":       { ... },                                    // optional
      "references": [ ... ],                                   // citation links, optional
      "context": "... surrounding sentence ...",
      "mentionContextAttributes":  { "used": {value, score}, "created": {...}, "shared": {...} },
      "documentContextAttributes": { "used": {value, score}, "created": {...}, "shared": {...} }
    }
  ]
}
```

We map it to:
- **papers** — one row per file.
- **mentions** — one row per mention.
- **purpose_assessments** — exactly **6 rows per mention**: scope `{local, document}` × purpose `{used, created, shared}`.

In [2]:
files = sorted(DATA_DIR.glob("*.software.json"))
print(f"Found {len(files)} Softcite files")

records = []
for f in files:
    with open(f, encoding="utf-8") as fh:
        records.append((f, json.load(fh)))

Found 265 Softcite files


In [3]:
def field(mention, key):
    """Return (raw, normalized) for an optional sub-object like version/publisher/url."""
    obj = mention.get(key) or {}
    return obj.get("rawForm"), obj.get("normalizedForm")


# --- papers table ---------------------------------------------------------
paper_rows = []
for f, d in records:
    paper_id = d.get("id") or f.name.replace(".software.json", "")
    biblio = d.get("biblio") or {}
    mentions = d.get("mentions", [])
    paper_rows.append({
        "paper_id": paper_id,
        "title": biblio.get("title"),
        "authors": biblio.get("authors"),
        "doi": biblio.get("doi"),
        "source_file_type": "tei",
        "n_mentions": len(mentions),
        "has_mentions": len(mentions) > 0,
        "file_name": f.name,
    })

papers = pd.DataFrame(paper_rows)
papers.head()

,paper_id,title,authors,doi,source_file_type,n_mentions,has_mentions,file_name
0,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,A Blockchain-Enhanced Reversible Watermarking ...,"Reda Bellafqira, Chloé Berton, Gouenou Coatrieux",10.1109/csp66295.2025.00016,tei,11,True,ABlockchainEnhancedReversibleWatermarkingFrame...
1,857559d553177a7f858abc48047c6f04209429e6,AI for Computational Biology: Highlights from ...,"M Ahmad, Michał Denkiewicz, Rishika Gupta, Sac...",NaN,tei,116,True,AIforComputationalBiologyHighlightsfromthefirs...
2,18fe244c5272cf018591691ea078a27ab937594f,A Standards-Based Knowledge Graph that Bridges...,"Marie Schmit, Ulysse Le Clanche, George Marchm...",NaN,tei,376,True,AStandardsBasedKnowledgeGraphthatBridgesScient...
3,ffecb02ef212e3593981ae9bb28d5a16ca1c4b86,A Survey on Multimodal Federated Learning,"Silvia Grosso, Nawel Benarba, Sara Bouchenak, ...",NaN,tei,49,True,ASurveyonMultimodalFederatedLearning.software....
4,72bd546fbc5c44c0be94925d3ac67eff54c9171c,NaN,NaN,NaN,tei,2,True,AWhiteBoxWatermarkingModulationforEncryptedDNN...


### The key reshape: melting purpose assessments

Softcite's `purpose_assessments` table is **tidy**: exactly six rows per mention, one for each `(scope, purpose)` pair, each with a boolean `value` and a `certainty_score`. Our raw JSON stores those same six numbers as **two nested blocks**:

| raw block | becomes `scope` |
|---|---|
| `mentionContextAttributes`  | `"local"`    |
| `documentContextAttributes` | `"document"` |

So tidying the data is a pure **unpivot (melt)**: nested-wide → long. No information is created or lost — just reshaped.

`melt_purposes` below does exactly this — a double loop over the two scopes and three purposes, returning a list of **6 dicts** with keys `software_mention_id`, `paper_id`, `mention_index`, `scope`, `purpose`, `value`, `certainty_score`. Two choices are baked in:

- The scope naming (`local` vs `document`) is the contract every later analysis depends on — Softcite's headline query filters `scope == "document"`.
- A missing block or purpose defaults to `value=False`, `score=NaN`, so absent data never masquerades as a real `0.0`.

In [4]:
SCOPES = {"local": "mentionContextAttributes", "document": "documentContextAttributes"}
PURPOSES = ("used", "created", "shared")


def melt_purposes(mention, paper_id, mention_index):
    """Unpivot a mention's two attribute blocks into 6 tidy purpose-assessment rows.

    Returns a list of exactly len(SCOPES) * len(PURPOSES) == 6 dicts.
    """
    sid = f"{paper_id}:tei:{mention_index}"
    rows = []
    for scope, block_key in SCOPES.items():
        attrs = mention.get(block_key) or {}
        for purpose in PURPOSES:
            a = attrs.get(purpose) or {}
            rows.append({
                "software_mention_id": sid,
                "paper_id": paper_id,
                "mention_index": mention_index,
                "scope": scope,
                "purpose": purpose,
                "value": bool(a.get("value", False)),
                "certainty_score": float(a["score"]) if "score" in a else float("nan"),
            })
    return rows


# Show the target shape on the first mention available.
first_mention = next(m for _f, _d in records for m in _d.get("mentions", []))
pd.DataFrame(melt_purposes(first_mention, "DEMO", 0))

,software_mention_id,paper_id,mention_index,scope,purpose,value,certainty_score
0,DEMO:tei:0,DEMO,0,local,used,False,9.250641e-05
1,DEMO:tei:0,DEMO,0,local,created,False,5.209446e-05
2,DEMO:tei:0,DEMO,0,local,shared,False,2.384186e-07
3,DEMO:tei:0,DEMO,0,document,used,True,9.998484e-01
4,DEMO:tei:0,DEMO,0,document,created,False,1.341581e-03
5,DEMO:tei:0,DEMO,0,document,shared,False,3.576279e-07


In [5]:
# --- mentions & purpose_assessments tables --------------------------------
mention_rows = []
purpose_rows = []
for f, d in records:
    paper_id = d.get("id") or f.name.replace(".software.json", "")
    for i, m in enumerate(d.get("mentions", [])):
        name = m.get("software-name") or {}
        lang = m.get("language") or {}
        v_raw, v_norm = field(m, "version")
        p_raw, p_norm = field(m, "publisher")
        u_raw, u_norm = field(m, "url")
        mention_rows.append({
            "software_mention_id": f"{paper_id}:tei:{i}",
            "paper_id": paper_id,
            "mention_index": i,
            "software_raw": name.get("rawForm"),
            "software_normalized": name.get("normalizedForm"),
            "software_type": m.get("software-type"),
            "version_raw": v_raw,
            "version_normalized": v_norm,
            "publisher_raw": p_raw,
            "publisher_normalized": p_norm,
            "language_raw": lang.get("rawForm"),
            "language_normalized": lang.get("normalizedForm"),
            "language_wikidata": lang.get("wikidataId"),
            "url_raw": u_raw,
            "url_normalized": u_norm,
            "context_full_text": m.get("context"),
            "n_references": len(m.get("references") or []),
        })
        purpose_rows.extend(melt_purposes(m, paper_id, i))

mentions = pd.DataFrame(mention_rows)
purpose_assessments = pd.DataFrame(purpose_rows)
mentions.head()

,software_mention_id,paper_id,mention_index,software_raw,software_normalized,software_type,version_raw,version_normalized,publisher_raw,publisher_normalized,language_raw,language_normalized,language_wikidata,url_raw,url_normalized,context_full_text,n_references
0,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:0,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,0,Alg,Alg,software,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Alg. 2 ensures perfect reconstruction of the o...,0
1,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:1,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,1,VGG16 2,VGG16 2,software,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,We utilized the VGG16 2 architecture as our ba...,0
2,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:2,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,2,VGG16,VGG16,software,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,We utilized a pre-trained VGG16 network fine-t...,0
3,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:3,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,3,VGG16,VGG16,software,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,We trained two versions of the VGG16 model: i)...,0
4,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:4,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,4,VGG16,VGG16,software,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Testing on a medical dataset using a VGG16 mod...,0


In [6]:
# --- sanity checks (the invariants Softcite's schema guarantees) ----------
print(f"papers:              {papers.shape}")
print(f"mentions:            {mentions.shape}")
print(f"purpose_assessments: {purpose_assessments.shape}")

assert len(purpose_assessments) == 6 * len(mentions), "expected exactly 6 purpose rows per mention"
assert mentions["software_mention_id"].is_unique, "software_mention_id must be unique"
per_mention = purpose_assessments.groupby("software_mention_id").size()
assert (per_mention == 6).all(), "every mention must have exactly 6 assessments"
print("\nAll invariants hold ✅")

papers:              (265, 8)
mentions:            (12115, 17)
purpose_assessments: (72690, 7)

All invariants hold ✅


### Explicit and implicit mentions

Softcite gives each mention a `software-type`:

- `software`, `component` (a plugin, a package) and `environment` (a programming language or environment): the software is **named** — these are the **explicit** mentions.
- `implicit`: the text refers to software without naming it ("our code", "the scripts", "a tool").

From here on, `mentions`, `purpose_assessments` and the counts in `papers` hold the **explicit** mentions only. The implicit ones are kept in `implicit_mentions` and `implicit_purposes` for section 7; `all_mentions` and `all_purpose_assessments` keep everything.

In [7]:
all_mentions, all_purpose_assessments = mentions, purpose_assessments
is_implicit = all_mentions["software_type"].eq("implicit")

implicit_mentions = all_mentions[is_implicit]
implicit_purposes = all_purpose_assessments[
    all_purpose_assessments["software_mention_id"].isin(implicit_mentions["software_mention_id"])]

# The analyses of sections 2–6 use the explicit mentions only.
mentions = all_mentions[~is_implicit]
purpose_assessments = all_purpose_assessments[
    all_purpose_assessments["software_mention_id"].isin(mentions["software_mention_id"])]

papers["n_mentions"] = papers["paper_id"].map(mentions.groupby("paper_id").size()).fillna(0).astype(int)
papers["has_mentions"] = papers["n_mentions"] > 0
papers["n_implicit"] = papers["paper_id"].map(implicit_mentions.groupby("paper_id").size()).fillna(0).astype(int)

print(all_mentions["software_type"].value_counts().to_string())
print(f"\nexplicit: {len(mentions)} mentions in {int(papers['has_mentions'].sum())} papers")
print(f"implicit: {len(implicit_mentions)} mentions in {int((papers['n_implicit'] > 0).sum())} papers")

software_type
software       11475
implicit         296
component        205
environment      139

explicit: 11819 mentions in 211 papers
implicit: 296 mentions in 122 papers


### One row per software and paper

Sections 2–6 work on the **software of each paper**, not on the mentions: a software named 30 times in a paper counts once. The `software` table has one row per paper and **normalized name** (Softcite's `normalizedForm`), with:

- `n_mentions`: how many times the paper mentions it, and `n_implicit_mentions`: how many implicit mentions were matched to it (next cell but one);
- `used`, `created`, `shared` and their `*_score`: Softcite's **document-level characterization** (`documentContextAttributes`). Softcite judges it over the whole paper, so every mention of a software in a paper carries the same values: the cell below checks it;
- `has_version`, `has_publisher`, `has_language`, `has_url`, `has_citation`: whether at least one of its mentions in the paper carries the attribute, and `language`, `publisher`: the first one given.

In [8]:
doc_scope = purpose_assessments[purpose_assessments["scope"] == "document"]
doc_values = doc_scope.pivot(index="software_mention_id", columns="purpose", values="value")
doc_scores = doc_scope.pivot(index="software_mention_id", columns="purpose", values="certainty_score").add_suffix("_score")
with_doc = mentions.join(doc_values, on="software_mention_id").join(doc_scores, on="software_mention_id")

keys = ["paper_id", "software_normalized"]
# The document-level characterization is the same for every mention of a software in a paper.
assert (with_doc.groupby(keys)[list(PURPOSES)].nunique() <= 1).all().all(), "document values differ within a paper"

any_ = lambda s: s.notna().any()
software = (with_doc.groupby(keys, as_index=False)
            .agg(n_mentions=("software_mention_id", "size"),
                 **{p: (p, "first") for p in PURPOSES},
                 **{f"{p}_score": (f"{p}_score", "first") for p in PURPOSES},
                 has_version=("version_normalized", any_),
                 has_publisher=("publisher_normalized", any_),
                 has_language=("language_normalized", any_),
                 has_url=("url_normalized", any_),
                 has_citation=("n_references", lambda s: (s > 0).any()),
                 language=("language_normalized", "first"),
                 publisher=("publisher_normalized", "first")))

papers["n_software"] = papers["paper_id"].map(software.groupby("paper_id").size()).fillna(0).astype(int)
print(f"software: {len(software)} (paper, software) rows, {software['software_normalized'].nunique()} distinct normalized names, "
      f"from {len(mentions)} explicit mentions")
software.head()

software: 2242 (paper, software) rows, 1620 distinct normalized names, from 11819 explicit mentions


,paper_id,software_normalized,n_mentions,used,created,shared,used_score,created_score,shared_score,has_version,has_publisher,has_language,has_url,has_citation,language,publisher
0,023222d011f63def7a90b3e720e9b036d0591c5c,BRAT-Eval,1,False,False,False,0.177453,0.000005,3.218651e-06,False,False,False,False,False,NaN,NaN
1,023222d011f63def7a90b3e720e9b036d0591c5c,NLStruct,4,False,False,True,0.170846,0.000005,9.724544e-01,True,False,False,False,False,NaN,NaN
2,04ec17ab053a13914640241c4d0468721d8f51b3,BioCypher,3,True,False,False,0.999986,0.001438,5.960464e-07,True,False,False,False,False,NaN,NaN
3,04ec17ab053a13914640241c4d0468721d8f51b3,DESeq2,2,True,False,False,0.999997,0.000002,3.576279e-07,True,False,False,False,False,NaN,NaN
4,04ec17ab053a13914640241c4d0468721d8f51b3,HIVE,9,True,False,False,0.999996,0.000317,4.768372e-07,False,False,False,False,False,NaN,NaN


### Implicit mentions matched to a software

An implicit mention does not name a software, so each one is counted on its own (section 7). The exception: when its normalized name is also the name of an **explicit software of the same paper**, it is matched to that software and counted with it (`n_implicit_mentions` in `software`), not in section 7.

In [9]:
explicit_keys = pd.MultiIndex.from_frame(software[keys])
is_matched = pd.MultiIndex.from_frame(implicit_mentions[keys]).isin(explicit_keys)
matched_implicit = implicit_mentions[is_matched]

# Section 7 counts the others, each mention on its own.
implicit_mentions = implicit_mentions[~is_matched]
implicit_purposes = implicit_purposes[implicit_purposes["software_mention_id"].isin(implicit_mentions["software_mention_id"])]
papers["n_implicit"] = papers["paper_id"].map(implicit_mentions.groupby("paper_id").size()).fillna(0).astype(int)

matched_counts = matched_implicit.groupby(keys).size().rename("n_implicit_mentions").reset_index()
software = software.merge(matched_counts, on=keys, how="left")
software["n_implicit_mentions"] = software["n_implicit_mentions"].fillna(0).astype(int)

print(f"{len(matched_implicit)} implicit mentions matched to an explicit software of their paper:")
print(matched_implicit["software_normalized"].value_counts().to_string())
print(f"\n{len(implicit_mentions)} implicit mentions left, each counted in section 7")

51 implicit mentions matched to an explicit software of their paper:
software_normalized
code       24
scripts    15
script      8
tool        2
medkit      1
library     1

245 implicit mentions left, each counted in section 7


## 2 — Dataset overview

In [10]:
overview = pd.Series({
    "papers": len(papers),
    "papers with >=1 explicit software": int((papers["n_software"] > 0).sum()),
    "papers with biblio (title)": int(papers["title"].notna().sum()),
    "papers with DOI": int(papers["doi"].notna().sum()),
    "distinct software (normalized names)": software["software_normalized"].nunique(),
    "(paper, software) pairs": len(software),
    "explicit mentions": len(mentions),
    "implicit mentions matched to a software of their paper": len(matched_implicit),
    "implicit mentions not matched (section 7)": len(implicit_mentions),
    "papers with >=1 unmatched implicit mention": int((papers["n_implicit"] > 0).sum()),
})
rate = (papers["n_software"] > 0).mean()
print(overview.to_string())
print(f"\nShare of papers naming at least one software: {rate:.1%}")
print(f"Software per paper, among those papers: median {papers.loc[papers['n_software'] > 0, 'n_software'].median():.0f}, "
      f"mean {papers.loc[papers['n_software'] > 0, 'n_software'].mean():.1f}")

papers                                                      265
papers with >=1 explicit software                           211
papers with biblio (title)                                  236
papers with DOI                                             114
distinct software (normalized names)                       1620
(paper, software) pairs                                    2242
explicit mentions                                         11819
implicit mentions matched to a software of their paper       51
implicit mentions not matched (section 7)                   245
papers with >=1 unmatched implicit mention                  119

Share of papers naming at least one software: 79.6%
Software per paper, among those papers: median 7, mean 10.6


In [11]:
# Distribution of distinct software per paper
per_paper = papers["n_software"].value_counts().reindex(range(papers["n_software"].max() + 1), fill_value=0)
fig = go.Figure(go.Bar(
    x=per_paper.index, y=per_paper.values, marker=dict(color=STYLE["series"][0], line=dict(color=STYLE["surface"], width=1)),
    hovertemplate="%{y} papers name %{x} distinct software<extra></extra>"))
fig.update_layout(title="Explicit software per paper (distinct normalized names)", bargap=0, height=380,
                  xaxis_title="distinct software named in a paper", yaxis_title="papers")
fig.update_xaxes(showgrid=False)
fig.show()

print(papers["n_software"].describe().to_string())

count    265.000000
mean       8.460377
std       10.735073
min        0.000000
25%        1.000000
50%        5.000000
75%       12.000000
max       61.000000


## 3 — Top software

Mirrors Softcite example #1. Software is identified by its **normalized** name, and ranked by the number of **papers** that name it: a software mentioned many times in one paper counts once.

In [12]:
TOP_N = 20

by_papers = software["software_normalized"].value_counts().head(TOP_N)
hbar(by_papers, f"Top {TOP_N} explicit software by papers", "papers").show()
by_papers

software_normalized
Python          41
Adam            17
Nextflow        17
Snakemake       13
PyTorch         13
Docker          12
code            12
XGBoost         11
Galaxy          11
scikit-learn    11
python          10
DESeq2           9
Seurat           9
Monolix          9
Linux            8
FedAvg           8
STAR             7
ChatGPT          7
scripts          6
WorkflowHub      6
Name: count, dtype: int64

## 4 — Purpose: the document characterization (the core analysis)

This mirrors Softcite example #3. For each software of a paper, Softcite says whether the paper **used**, **created** or **shared** it, judged over the whole paper (`documentContextAttributes`), with a certainty score. A software can have several purposes, or none.

Softcite's headline query keeps the assessments with a score above 0.5. In this output, that is exactly `value == True` (the cell below checks it).

In [13]:
# How many (paper, software) pairs Softcite characterizes as used / created / shared.
# Softcite's value is True exactly when its score is above 0.5.
assert all(((software[f"{p}_score"] > 0.5) == software[p]).all() for p in PURPOSES)
characterized = pd.Series({p: int(software[p].sum()) for p in PURPOSES}, name="(paper, software) pairs")
share = characterized / len(software) * 100

fig = go.Figure(go.Bar(
    x=share.values[::-1], y=list(share.index[::-1]), orientation="h", marker_color=STYLE["series"][0],
    customdata=characterized.values[::-1], text=[f"{v:.1f}%" for v in share.values[::-1]],
    textposition="outside", textfont=dict(color=STYLE["ink2"]), cliponaxis=False,
    hovertemplate="<b>%{y}</b><br>%{customdata:,d} (paper, software) pairs, %{x:.1f}%<extra></extra>"))
fig.update_layout(title=f"Document characterization of the {len(software):,} (paper, software) pairs",
                  xaxis_title="% of (paper, software) pairs", height=210)
fig.update_xaxes(range=[0, 100])
fig.update_yaxes(showgrid=False, tickfont=dict(color=STYLE["ink2"]))
fig.show()
characterized

used       1566
created     403
shared      143
Name: (paper, software) pairs, dtype: int64

In [14]:
def top_software_for(purpose, min_score=0.5, top_n=15):
    """Papers per software where the document characterization says `purpose` with a score above min_score.

    Replicates Softcite example #3:  scope == 'document', purpose == 'used', score > 0.5.
    """
    hits = software[software[purpose] & (software[f"{purpose}_score"] > min_score)]
    return hits["software_normalized"].value_counts().head(top_n)


popular_used = top_software_for("used")
hbar(popular_used, "Most-used explicit software (document characterization, score > 0.5)", "papers").show()
popular_used

software_normalized
Python          29
Adam            16
Nextflow        13
PyTorch         10
XGBoost          9
python           9
Monolix          9
code             9
DESeq2           8
FedAvg           8
scikit-learn     8
Seurat           7
Snakemake        7
STAR             7
scripts          6
Name: count, dtype: int64

In [15]:
# Software the papers' authors created, and software they shared
print("Top software CREATED (document characterization, score > 0.5), by papers:")
print(top_software_for("created").to_string())
print("\nTop software SHARED (document characterization, score > 0.5), by papers:")
print(top_software_for("shared").to_string() or "(none above threshold)")

Top software CREATED (document characterization, score > 0.5), by papers:
software_normalized
Nextflow       12
Python         10
FedAvg          6
Snakemake       6
code            6
XGBoost         4
Galaxy          4
scripts         4
BoNesis         3
EDAM            3
PyTorch         3
Hyperledger     3
Docker          3
Mapper          3
git             2

Top software SHARED (document characterization, score > 0.5), by papers:
software_normalized
code            5
Python          4
NeurIPS         4
NLStruct        2
gitOmmix        2
MultiQC         2
Software        2
Nextflow        2
NeuroConText    2
ktest           2
medkit          2
BoNesis         2
AstroLogics     1
function        1
scripts         1


## 5 — Metadata richness

Beyond the name, the mentions of a software may carry a programming **language** (with a Wikidata id), a **publisher**, a **version**, a **URL**, and links to the paper's **citations**. For how many (paper, software) pairs does at least one mention carry each?

In [16]:
richness = pd.Series({
    "has version": software["has_version"].mean(),
    "has publisher": software["has_publisher"].mean(),
    "has language": software["has_language"].mean(),
    "has url": software["has_url"].mean(),
    "has >=1 citation": software["has_citation"].mean(),
}) * 100
hbar(richness.sort_values(ascending=False), f"Share of the {len(software):,} (paper, software) pairs carrying each attribute",
     "of the (paper, software) pairs", fmt=".1f", suffix="%", labels=True,
     x_title="% of (paper, software) pairs").show()
richness.round(1)

has version         10.3
has publisher        2.0
has language         1.2
has url              2.5
has >=1 citation    13.9
dtype: float64

In [17]:
# Counted once per (paper, software)
lang = software["language"].value_counts().head(10)
pub = software["publisher"].value_counts().head(10)
hbar_pair(lang, pub, ("Top programming languages", "Top publishers"),
          ("(paper, software) pairs", "(paper, software) pairs")).show()

## 6 — Single-software deep dive

Mirrors Softcite example #1 ("how many papers mention X?"). Change `SOFTWARE` to any normalized name that appears in section 3.

In [18]:
# Default to the software named in the most papers, so the cell always has something to show.
SOFTWARE = software["software_normalized"].value_counts().index[0]

rows = software[software["software_normalized"] == SOFTWARE]
print(f"Software: {SOFTWARE!r}")
print(f"  papers:   {len(rows)}")
print(f"  mentions: {int(rows['n_mentions'].sum())}")
print("  document characterization (papers): " + ", ".join(f"{p} {int(rows[p].sum())}" for p in PURPOSES))
print("\nSample contexts:")
for ctx in mentions.loc[mentions["software_normalized"] == SOFTWARE, "context_full_text"].dropna().head(3):
    print("  •", (ctx[:240] + "…") if len(ctx) > 240 else ctx)

Software: 'Python'
  papers:   41
  mentions: 125
  document characterization (papers): used 29, created 10, shared 4

Sample contexts:
  • Finally, the team plans to containerize the pipeline as a Python package, which mirrors Team 1's and Team 4's emphasis on reproducibility and usability, pointing toward a future in which multi-omic data analysis can be deployed modularly ac…
  • Team 7 proposed packaging their work into a standardized and reusable Python module, enabling wider dissemination and reproducibility.
  • Expression patterns of phase-determining genes were visualized through ridge plots using Seaborn 0.13.2 visualization libraries in Python 3.11.


## 7 — Implicit mentions

Implicit mentions refer to software without naming it: "the code is available on GitHub", "we wrote scripts". They matter for the *created* and *shared* questions — a paper that shares its own code often never names it.

Every implicit mention counts on its own: it does not name a software, so it cannot be grouped by software like the explicit ones. Only the implicit mentions whose normalized name is an explicit software of the same paper are left out: they were matched to that software in section 1.

Two views:

- **by mentions and by documents:** the words used, counted per mention and per distinct paper. Case is ignored (`Code` and `code` are one form).
- **by mention scope and document scope:** how often Softcite flags each purpose (used / created / shared) for these mentions, judged from the mention's own context (`local`) or from the whole paper (`document`).

In [19]:
implicit_forms = implicit_mentions.assign(form=implicit_mentions["software_normalized"].str.lower().str.strip())

print(f"{len(implicit_mentions)} implicit mentions (not matched to a software) in {implicit_mentions['paper_id'].nunique()} papers "
      f"({implicit_mentions['paper_id'].nunique() / len(papers):.1%} of the papers), "
      f"{implicit_forms['form'].nunique()} distinct forms")

forms_by_mentions = implicit_forms["form"].value_counts().head(TOP_N)
forms_by_papers = (implicit_forms.groupby("form")["paper_id"].nunique()
                   .sort_values(ascending=False).head(TOP_N))

hbar_pair(forms_by_mentions, forms_by_papers,
          (f"Top {TOP_N} implicit forms by mentions", f"Top {TOP_N} implicit forms by papers"),
          ("mentions", "papers")).show()

245 implicit mentions (not matched to a software) in 119 papers (44.9% of the papers), 28 distinct forms


In [20]:
# Purposes flagged True for the implicit mentions, at mention (local) and document scope
implicit_fired = (implicit_purposes[implicit_purposes["value"]]
                  .groupby(["scope", "purpose"]).size().unstack("purpose")
                  .reindex(index=["local", "document"], columns=list(PURPOSES)).fillna(0).astype(int))


# The same as a share, next to the explicit mentions, since there are far fewer implicit ones
def share_fired(assessments, n):
    return (assessments[assessments["value"]].groupby(["scope", "purpose"]).size() / n * 100).unstack("purpose") \
        .reindex(index=["local", "document"], columns=list(PURPOSES)).fillna(0)


SCOPE_LABELS = {"local": "mention (local)", "document": "document"}
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.06,
                    subplot_titles=(f"Implicit mentions (n={len(implicit_mentions):,})",
                                    f"Explicit mentions (n={len(mentions):,})"))
for col, (assessments, n) in enumerate(((implicit_purposes, len(implicit_mentions)),
                                        (purpose_assessments, len(mentions))), start=1):
    share = share_fired(assessments, n)
    for k, purpose in enumerate(PURPOSES):
        fig.add_trace(go.Bar(
            x=share[purpose].values, y=[SCOPE_LABELS[s] for s in share.index], orientation="h",
            name=purpose, legendgroup=purpose, showlegend=col == 1, marker_color=STYLE["series"][k],
            hovertemplate=f"<b>{purpose}</b> · %{{y}}<br>%{{x:.1f}}% of the mentions<extra></extra>"), row=1, col=col)
fig.update_layout(title="Purposes flagged for each kind of mention, by scope", barmode="group", bargroupgap=0.08,
                  height=360, margin=dict(t=96), legend=dict(traceorder="normal"))
fig.update_xaxes(title_text="% of mentions flagged True", range=[0, 100])
fig.update_yaxes(showgrid=False, tickfont=dict(color=STYLE["ink2"]), autorange="reversed")
fig.update_annotations(font=dict(size=14, color=STYLE["ink"]))
fig.show()
implicit_fired

purpose,used,created,shared
scope,,,
local,97,21,83
document,131,30,100


## 8 — Limitations

- **No publication year / genre / license.** The raw Softcite output carries only `biblio` (title, authors, doi), so Softcite's *papers-per-year trend* example and any genre/license breakdown are not reproducible from this data alone. Joining on DOI against an external metadata source (e.g. Crossref / OpenAlex) would be needed to recover publication dates.
- **Probabilistic extractions.** These mentions come from an ML model (Softcite reports f-scores around 0.8), so expect both false positives and false negatives. The `certainty_score` thresholds (e.g. `> 0.5`) trade recall for precision.
- **Explicit vs implicit.** The split relies on Softcite's `software-type`, which is itself a prediction. Implicit forms are generic words, so they say little about *which* software; read them for the purposes (created, shared) rather than the names.
- **Mixed inputs.** Most files come from the GROBID TEI (237 of the 265 have `biblio`); the others were run on plain text and have no bibliographic metadata, so their title, authors and DOI are empty.
- **Normalized names.** Software is identified by Softcite's `normalizedForm`, which is not case-folded: 85 (paper, software) pairs differ from another one of the same paper only by case (`TensorFlow` / `Tensorflow`), so such a software is counted twice in its paper. Some entries are extraction noise rather than named software (algorithms such as `Adam`, repositories such as `CRAN`).
- **Document characterization.** Sections 4–6 use Softcite's document-level `used` / `created` / `shared`, which is the same for every mention of a software in a paper. The mention-level (`local`) assessments are only used in section 7, to compare implicit and explicit mentions.